# Invoice DocAI v2 -- 02 Baseline OCR

Run EasyOCR on all 347 SROIE validation documents, extract vendor / date / total
with the **improved v2 rule-based extractor**, compute field-level metrics, and
analyse errors.

**Plan**
1. Resolve project root, import `docai_utils` (v2)
2. Install EasyOCR if needed
3. Run OCR inference on ALL val docs (with checkpointing every 50 docs)
4. Compute metrics (P / R / F1 per field + micro)
5. Latency statistics
6. Visualisation: bar chart, error distribution, text summary
7. Error analysis: top-20 worst documents

## 1. Setup & imports

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# Mount Google Drive (Colab)
# ---------------------------------------------------------------------------
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# ---------------------------------------------------------------------------
# Resolve PROJECT_ROOT
# ---------------------------------------------------------------------------
def _is_project_root(p: Path) -> bool:
    """A directory is the project root if it contains data/sroie/processed."""
    return (p / 'data' / 'sroie' / 'processed').exists()


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        cwd / 'invoice_docai',
        # Google Drive locations
        Path('/content/drive/MyDrive/ORC/invoice_docai'),
        Path('/content/drive/MyDrive/invoice_docai'),
        Path('/content/drive/MyDrive/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/Yandex.Disk/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/invoice_docai'),
        # Local Windows
        Path(r'c:\Yandex.Disk\Yandex.Disk\ML Neimark\From OCR 2 Transformers\invoice_docai'),
    ]
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if _is_project_root(p):
            return p
        child = p / 'invoice_docai'
        if child.is_dir() and _is_project_root(child):
            return child
    raise FileNotFoundError(
        'Cannot locate invoice_docai project root (must contain data/sroie/processed)'
    )


PROJECT_ROOT = resolve_project_root()

# ---------------------------------------------------------------------------
# Determine SRC_ROOT and OUTPUT_ROOT
# v2/ subfolder takes priority if it exists
# ---------------------------------------------------------------------------
if (PROJECT_ROOT / 'v2' / 'src').exists():
    SRC_ROOT = PROJECT_ROOT / 'v2' / 'src'
else:
    SRC_ROOT = PROJECT_ROOT / 'src'

if (PROJECT_ROOT / 'v2').exists():
    OUTPUT_ROOT = PROJECT_ROOT / 'v2' / 'outputs'
else:
    OUTPUT_ROOT = PROJECT_ROOT / 'outputs'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# ---------------------------------------------------------------------------
# Import from docai_utils (v2)
# ---------------------------------------------------------------------------
from docai_utils import (
    extract_fields_from_lines,
    evaluate,
    normalize_date,
    normalize_total,
    normalize_vendor,
)

# ---------------------------------------------------------------------------
# Load manifest
# ---------------------------------------------------------------------------
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'sroie' / 'processed'
manifest = pd.read_csv(PROCESSED_ROOT / 'manifest_val.csv')
manifest = manifest.dropna(subset=['image_path']).reset_index(drop=True)

print(f'PROJECT_ROOT  = {PROJECT_ROOT}')
print(f'SRC_ROOT      = {SRC_ROOT}')
print(f'OUTPUT_ROOT   = {OUTPUT_ROOT}')
print(f'Val samples   = {len(manifest)}')

## 2. Install EasyOCR

In [ ]:
try:
    import easyocr
    print('EasyOCR already installed')
except ImportError:
    print('Installing EasyOCR ...')
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, '-m', 'pip', 'install', 'easyocr', '-q'])
    import easyocr
    print('EasyOCR installed')

# Detect GPU availability
import torch
_use_gpu = torch.cuda.is_available()
print(f'GPU available: {_use_gpu}')

## 3. OCR inference (all val documents)

In [ ]:
import easyocr
import torch

_use_gpu = torch.cuda.is_available()
reader = easyocr.Reader(['en'], gpu=_use_gpu)
print(f'EasyOCR reader initialised (gpu={_use_gpu})')

N_DOCS = len(manifest)
CHECKPOINT_EVERY = 50
CHECKPOINT_PATH = OUTPUT_ROOT / 'ocr_predictions_checkpoint.csv'
FINAL_PATH = OUTPUT_ROOT / 'ocr_predictions_clean.csv'

# ---------------------------------------------------------------------------
# Resume from checkpoint if available
# ---------------------------------------------------------------------------
done_ids = set()
rows = []
latencies = []

if CHECKPOINT_PATH.exists():
    _ckpt = pd.read_csv(CHECKPOINT_PATH)
    done_ids = set(_ckpt['id'].tolist())
    rows = _ckpt.to_dict('records')
    print(f'Resumed from checkpoint: {len(done_ids)} docs already processed')

# ---------------------------------------------------------------------------
# Run inference
# ---------------------------------------------------------------------------
for idx, row in tqdm(manifest.iterrows(), total=N_DOCS, desc='OCR inference'):
    doc_id = row['id']
    if doc_id in done_ids:
        continue

    img_path = row['image_path']

    start = time.perf_counter()
    try:
        ocr_out = reader.readtext(img_path, detail=1, paragraph=False)
        lines = [item[1] for item in ocr_out]
    except Exception as e:
        print(f'  [WARN] OCR failed for {doc_id}: {e}')
        lines = []
    elapsed_ms = (time.perf_counter() - start) * 1000

    pred = extract_fields_from_lines(lines)

    rows.append({
        'id': doc_id,
        'pred_vendor': pred['pred_vendor'],
        'pred_date': pred['pred_date'],
        'pred_total': pred['pred_total'],
        'latency_ms': elapsed_ms,
    })
    latencies.append(elapsed_ms)
    done_ids.add(doc_id)

    # Checkpoint every N docs
    if len(done_ids) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(rows).to_csv(CHECKPOINT_PATH, index=False)
        print(f'  checkpoint saved ({len(done_ids)} / {N_DOCS})')

# ---------------------------------------------------------------------------
# Save final predictions
# ---------------------------------------------------------------------------
pred_df = pd.DataFrame(rows)
pred_df.to_csv(FINAL_PATH, index=False)
print(f'\nSaved {len(pred_df)} predictions to {FINAL_PATH}')

# Clean up checkpoint
if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print('Checkpoint file removed (final file saved)')

## 4. Metrics

In [ ]:
# Load predictions (works even if inference cell was run in a previous session)
pred_df = pd.read_csv(OUTPUT_ROOT / 'ocr_predictions_clean.csv')

gt_df = manifest[['id', 'gt_vendor', 'gt_date', 'gt_total']].copy()
metrics_df, errors_df = evaluate(gt_df, pred_df)

display(metrics_df)

## 5. Latency statistics

In [ ]:
# Use latency column from predictions if available, else from in-memory list
if 'latency_ms' in pred_df.columns:
    lat = pred_df['latency_ms'].dropna().values
elif latencies:
    lat = np.array(latencies)
else:
    lat = np.array([])

if len(lat) > 0:
    print(f'Latency  mean : {np.mean(lat):,.1f} ms/doc')
    print(f'Latency  p50  : {np.percentile(lat, 50):,.1f} ms/doc')
    print(f'Latency  p90  : {np.percentile(lat, 90):,.1f} ms/doc')
else:
    print('No latency data available (latency_ms column missing from predictions file).')

## 6. Visualisation

In [ ]:
import matplotlib.pyplot as plt

field_metrics = metrics_df[metrics_df['field'].isin(['vendor', 'date', 'total'])].copy()

# --- 6a. Bar chart: P / R / F1 by field -----------------------------------
plot_df = field_metrics.set_index('field')[['precision', 'recall', 'f1']]
ax = plot_df.plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0,
                  color=['#4C78A8', '#F58518', '#54A24B'])
ax.set_title('Baseline OCR v2: field-level P / R / F1')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.25)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

# --- 6b. Distribution of wrong fields per document -------------------------
wrong_counts = errors_df['num_wrong_fields'].value_counts().sort_index()
ax2 = wrong_counts.plot(kind='bar', figsize=(7, 4), color='#4C78A8', rot=0)
ax2.set_title('Documents by number of wrong fields')
ax2.set_xlabel('Wrong fields per document')
ax2.set_ylabel('Document count')
ax2.grid(axis='y', alpha=0.25)
for container in ax2.containers:
    ax2.bar_label(container, fontsize=9)
plt.tight_layout()
plt.show()

# --- 6c. Text summary ------------------------------------------------------
micro_row = metrics_df[metrics_df['field'] == 'micro'].iloc[0]
best = field_metrics.sort_values('f1', ascending=False).iloc[0]
worst = field_metrics.sort_values('f1', ascending=True).iloc[0]

print(f"Micro F1 : {micro_row['f1']:.3f}  |  Precision : {micro_row['precision']:.3f}  |  Recall : {micro_row['recall']:.3f}")
print(f"Best field  by F1 : {best['field']}  ({best['f1']:.3f})")
print(f"Worst field by F1 : {worst['field']} ({worst['f1']:.3f})")

## 7. Error analysis (top 20 worst documents)

In [ ]:
display_cols = [
    'id',
    'gt_vendor',  'pred_vendor',
    'gt_date',    'pred_date',
    'gt_total',   'pred_total',
    'num_wrong_fields',
]

top_errors = errors_df.sort_values('num_wrong_fields', ascending=False).head(20)
display(top_errors[display_cols].reset_index(drop=True))